In [15]:
# imports
from sklearn.pipeline import Pipeline
from joblib import Memory
from utils.pipeline import *
from utils.fx import FXRateDownloader
from utils.config import *
import pandas as pd

# Saving intermediary datasets for inspection

## Preprocessing Pipeline

In [16]:
# # Full pipeline with debug steps after each transformer
# preprocessing_pipeline = Pipeline([
#     ("column_renamer", ColumnRenamer(rename_dict=COLUMN_RENAME_DICT)),
#     ("debug_renamed", DebugTransformer("after renaming")),

#     ("duplicate_dropper", DuplicateDropper(duplicate_col="name")),
#     ("debug_duplicates_dropped", DebugTransformer("after duplicate drop")),

#     ("column_dropper_1", ColumnDropper(cols_to_drop=COLUMNS_TO_DROP_1)),
#     ("debug_droppedcols_1", DebugTransformer("after column drop 1")),

#     ("prefilterer", Prefilterer(
#         filter_map={
#             "round_types": ["SPINOUT"]		# dont include spinouts, since they have different growth dynamic
#         })
#     ),
#     ("debug_prefiltered", DebugTransformer("after prefiltering")),

#     ("list_parser", ListParser()),
#     ("debug_parsedlists", DebugTransformer("after list parsing")), # keep debugger because last step is not cached
# ], verbose=True)

# # Load raw data
# data_loader = DataLoader(file_path="data/dr/raw/dr_raw.pkl")#, n_rows=100)
# data_loader.fit()
# raw_data = data_loader.transform(X=None)

# # Fit preprocessing_pipeline on raw data
# preprocessed_data = preprocessing_pipeline.fit_transform(raw_data)

# # Inspect debug
# renamed_data = preprocessing_pipeline.named_steps["debug_renamed"].output_
# deduped_data = preprocessing_pipeline.named_steps["debug_duplicates_dropped"].output_
# columns_dropped_1_data = preprocessing_pipeline.named_steps["debug_droppedcols_1"].output_
# prefiltered_data = preprocessing_pipeline.named_steps["debug_prefiltered"].output_
# parsedlists_data = preprocessing_pipeline.named_steps["debug_parsedlists"].output_

In [ ]:
import pandas as pd
import pickle

with open('data/dr/dr_preprocessed.pkl', 'rb') as f:
    preprocessed_data = pickle.load(f)
preprocessed_data

In [ ]:
filter = preprocessed_data[preprocessed_data['id'] == '4046245']
filter

## Save preprocessed data

In [19]:
# preprocessed_data.to_pickle('data/dr/dr_preprocessed.pkl')

## Download FX rates

In [20]:
# # This only happens once or if the fx cache is missing
# fx_downloader = FXRateDownloader(
#     base_currency="EUR",
#     target_currency="USD",
#     cache_file="data/fx_rates.pkl"
# )
# fx_downloader.download_for_df(preprocessed_data)  # writes cache file

In [21]:
# preprocessed_data = preprocessed_data.head(10)

In [22]:
# Success window for evaluating cumulative funding, employees, and valuation
WINDOW_YEARS = 3

buyin_round_extractor = BuyInRoundExtractor(
        fx_cache_file="data/fx_rates.pkl",
        buyin_round_number = 1
)

target_extractor = TargetExtractor(
        fx_cache_file="data/fx_rates.pkl",
        window_years=WINDOW_YEARS,
        download_date=DOWNLOAD_DATE,
        exit_round_types=EXIT_ROUND_TYPES
)

cat_encoder = CatEncoder(
    cat_columns=CAT_COLUMNS,
    cat_list_columns=CAT_LIST_COLUMNS,
    count_list_columns=COUNT_LIST_COLUMNS,
    count_lol_columns=COUNT_LOL_COLUMNS,
    remove_cat=REMOVE_CAT,
    remove_string=REMOVE_STRING,
    remove_cat_with_string=REMOVE_CAT_WITH_STRING,
    min_frequency=100
)

encoder = ColumnTransformer(
    transformers=[
        (
            "cat_encoder",
            cat_encoder,
            CAT_COLUMNS
            + CAT_LIST_COLUMNS
            + COUNT_LIST_COLUMNS
            + COUNT_LOL_COLUMNS
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False
)
encoder.set_output(transform="pandas")

parsing_pipeline = Pipeline([
    # Extract engineered features
    ("feature_extractor", FeatureExtractor()),
    ("debug_feature_extraction", DebugTransformer("after feature extraction")),
    
    # Extract buy-in round
    ("buyin_round_extractor", buyin_round_extractor),
    ("debug_buyin_round", DebugTransformer("after buyin round extraction")),

    # Expand events and compute forward-looking targets
    ("target_extractor", target_extractor),
    ("debug_targets", DebugTransformer("after target extraction")),
    
    ("quantile_binarizer", QuantileTargetBinarizer(
                quantile_threshold=0.75,
            )),
    ("debug_binarizer", DebugTransformer("after target binarization")),
    
    # Merge textual embedding and/or scores
    ("embedding_score_merger", EmbeddingScoreMerger(
        input_type='emb_score',
        embedding_paths={
            "qwen3": "data/dr/text/embeddings/qwen3-embedding_latest.pkl",
            "gemma": "data/dr/text/embeddings/embeddinggemma_latest.pkl"},
        embedding_model='qwen3',
        embedding_pca_dim=50,
        scoring_paths={'gemma3': 'data/dr/text/scoring/local/gemma3_temp0_scores_final.pkl',
                       'gpt-4o': 'data/dr/text/scoring/azure/gpt-4o_temp0_scores_final.pkl',
                       'kimi2.5': 'data/dr/text/scoring/azure/kimi-k2.5_temp0_scores_final.pkl'},
        scoring_model="avg"
    )),
    ("debug_emb_score", DebugTransformer("after embedding/score merge")),
    
    # Merge macroeconomic data (depends on buyin date column from buyin_round_extractor)
    ("ten_year_tby_merger", TenYearTBYMerger(
        file_path="data/macroeconomy/10_year_bond_yield_clean.csv",
        columns_to_merge={
            "US 10 Year": "ten_year_tby_us",
            "EU 10 Year": "ten_year_tby_eu"
        },
        merge=True
    )),
    ("debug_tby", DebugTransformer("after tby merge")),

    ("encoding", encoder),
    ("debug_encoded", DebugTransformer("after encoding")),
    
    # Drop unnecessary columns
    ("column_dropper_2", ColumnDropper(cols_to_drop=COLUMNS_TO_DROP_2)),
    ("debug_droppedcols_2", DebugTransformer("after column drop 2")),
], verbose=True)

# Fit parsing pipeline on preprocessed data
parsed_data = parsing_pipeline.fit_transform(preprocessed_data)

features_extracted = parsing_pipeline.named_steps["debug_feature_extraction"].output_
buyin_round_extracted = parsing_pipeline.named_steps["debug_buyin_round"].output_
target_extracted = parsing_pipeline.named_steps["debug_targets"].output_
target_binarized = parsing_pipeline.named_steps["debug_binarizer"].output_
emb_score_merged = parsing_pipeline.named_steps["debug_emb_score"].output_
tby_merged = parsing_pipeline.named_steps["debug_tby"].output_
encoded_data = parsing_pipeline.named_steps["debug_encoded"].output_
columns_dropped_2_data = parsing_pipeline.named_steps["debug_droppedcols_2"].output_

[Pipeline]  (step 1 of 16) Processing feature_extractor, total=   0.5s
   - debugger output shape: (8738, 85)
[Pipeline]  (step 2 of 16) Processing debug_feature_extraction, total=   0.0s
[Pipeline]  (step 3 of 16) Processing buyin_round_extractor, total=   1.0s
   - debugger output shape: (8738, 98)
[Pipeline]  (step 4 of 16) Processing debug_buyin_round, total=   0.0s
   - censored rows removed: 1696 out of 8738
[Pipeline] . (step 5 of 16) Processing target_extractor, total=  20.3s
   - debugger output shape: (7042, 104)
[Pipeline] .... (step 6 of 16) Processing debug_targets, total=   0.0s
   - quantile thresholds fitted: {'target_funding_delta': np.float64(4.99375112987563), 'target_valuation_delta': np.float64(23.03621444201313), 'target_employee_delta': np.float64(19.369661266568485)}
[Pipeline]  (step 7 of 16) Processing quantile_binarizer, total=   0.0s
   - debugger output shape: (7042, 107)
[Pipeline] .. (step 8 of 16) Processing debug_binarizer, total=   0.0s
[Pipeline]  (st

In [23]:
parsed_data

,hq_country_Belgium,hq_country_Denmark,hq_country_Finland,hq_country_Germany,hq_country_Netherlands,hq_country_Norway,hq_country_Sweden,industries_education,industries_energy,industries_enterprise software,...,qwen3_embedding_pca_42_of_50,qwen3_embedding_pca_43_of_50,qwen3_embedding_pca_44_of_50,qwen3_embedding_pca_45_of_50,qwen3_embedding_pca_46_of_50,qwen3_embedding_pca_47_of_50,qwen3_embedding_pca_48_of_50,qwen3_embedding_pca_49_of_50,ten_year_tby_eu,ten_year_tby_us
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.006585,0.069046,0.043640,-0.090009,-0.012825,-0.014684,-0.003390,-0.012326,3.77683,4.79062
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.062044,0.012039,-0.010325,-0.080343,-0.047743,0.035062,-0.003091,0.034917,4.01097,3.66009
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.01097,3.66009
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.015552,-0.017316,-0.013064,0.048952,-0.069999,0.004192,0.071729,0.098952,2.77140,3.18534
4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.77140,3.18534
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7037,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.077872,0.009139,0.088387,-0.032054,-0.011527,-0.019709,0.000612,-0.003090,2.46441,3.96964
7038,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.032379,0.022420,0.045363,0.004692,0.009442,0.060763,-0.034980,-0.045239,2.46441,3.96964
7039,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.104161,-0.016380,-0.023428,0.024353,-0.041570,0.014282,0.011614,0.059004,2.46441,3.96964
7040,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.042185,-0.017525,0.010326,-0.009783,-0.049343,-0.018698,-0.004936,-0.015769,2.46441,3.96964


# Checks

### Check round and event expansion

In [ ]:
rounds = buyin_round_extractor.expand_rounds(preprocessed_data)
rounds

In [ ]:
buyin_rounds = buyin_round_extractor.fit_transform(features_extracted)
buyin_rounds

In [ ]:
events = target_extractor.expand_events(buyin_rounds)
events

In [ ]:
targets = target_extractor.fit_transform(buyin_rounds)
targets

In [28]:
parsed_data.to_pickle(f'data/dr/dr_parsed_window_{WINDOW_YEARS}.pkl')